# 1. Environment Setting and Libraries

In [ ]:
import re
import os, json

# dedent : 여러줄 문자열을 코드상 보기 좋게 들여쓰기했을 때, 공통 들여쓰기를 제거하는 함수 (불필요한 앞쪽 공백 제거)
from textwrap import dedent
from pprint import pprint

import warnings

warnings.filterwarnings("ignore")

from dotenv import load_dotenv

load_dotenv()

##### google 사용가능한 모델 리스트

- models/gemini-2.5-flash
- models/gemini-2.5-pro
- models/gemini-2.5-flash-preview-tts
- models/gemini-2.5-pro-preview-tts
- models/gemma-4-26b-a4b-it
- models/gemma-4-31b-it
- models/gemini-flash-latest
- models/gemini-flash-lite-latest
- models/gemini-pro-latest
- models/gemini-2.5-flash-lite
- models/gemini-2.5-flash-image
- models/gemini-3-flash-preview
- models/gemini-3.1-pro-preview
- models/gemini-3.1-pro-preview-customtools
- models/gemini-3.1-flash-lite-preview
- models/gemini-3.1-flash-lite
- models/gemini-3-pro-image-preview
- models/gemini-3-pro-image
- models/nano-banana-pro-preview
- models/gemini-3.1-flash-image-preview
- models/gemini-3.1-flash-image
- models/gemini-3.1-flash-lite-image
- models/gemini-3.5-flash
- models/gemini-3.5-flash-lite
- models/gemini-omni-flash-preview
- models/gemini-omni-1.1-flash
- models/gemini-3.5-transcribe
- models/gemini-3.6-flash
- models/gemini-3.7-flash
- models/gemini-3.8-flash
- models/lyria-3-clip-preview
- models/lyria-3-pro-preview
- models/lyria-3.5
- models/gemini-3.1-flash-tts-preview
- models/gemini-3.8-flash-tts
- models/gemini-3.8-flash-lite-tts
- models/gemini-robotics-er-2-preview
- models/gemini-2.5-computer-use-preview-10-2025
- models/antigravity-preview-05-2026
- models/antigravity-preview-09-2026
- models/antigravity-preview-latest
- models/deep-research-max-preview-04-2026
- models/deep-research-preview-04-2026
- models/deep-research-pro-preview-12-2025
- models/gemini-embedding-001
- models/gemini-embedding-2-preview
- models/gemini-embedding-2
- models/aqa
- models/veo-3.1-generate-preview
- models/veo-3.1-fast-generate-preview
- models/veo-3.1-lite-generate-preview
- models/gemini-3.5-transcribe-live
- models/gemini-2.5-flash-native-audio-latest
- models/gemini-2.5-flash-native-audio-preview-09-2025
- models/gemini-2.5-flash-native-audio-preview-12-2025
- models/gemini-3.1-flash-live-preview
- models/gemini-3.8-live
- models/gemini-3.8-live-extended-thinking
- models/gemini-robotics-er-2-streaming-preview
- models/gemini-3.5-live-translate-preview
- models/lyria-realtime-exp

# 2. Tool Calling
- llm can use the external resources like other app's API

### 2-1. Tavily Search (Example)

(1) Define Tool

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch

In [ ]:
# set query for search
query = "스테이크와 어울리는 와인을 추천해주세요."

# initialize web search tool
web_search = TavilySearch(max_results=2, include_answer=True)

# print(type(web_search))
# print(web_search.name)
# pprint(web_search.description)
# pprint(web_search.args_schema.schema())

# execute web search
search_results = web_search.invoke({"query": query})

# print search results
for key, value in search_results.items():
    if key == "answer" or key == "results":
        print(key, " : ", value)

In [ ]:
# 도구 속성
print("자료형: ")
print(type(web_search))
print("-"*100)

print("name: ")
print(web_search.name)
print("-"*100)

print("description: ")
pprint(web_search.description)
print("-"*100)

print("schema: ")
pprint(web_search.args_schema.schema())
print("-"*100)

(2) tool calling

In [ ]:
# initialize GEMINI model
gemini_model = ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0.0)

# initialize web search tool
web_search = TavilySearch(max_results=2)

# binding tool with llm model
gemini_with_tools = gemini_model.bind_tools(tools = [web_search])

In [ ]:
# case : tool calling not required 
query = "Hello."
ai_msg = gemini_with_tools.invoke(query)

# print results
pprint(ai_msg)
print("="*50)

# print content of ai_msg
pprint(ai_msg.content)
print("="*50)

# print tool calls
pprint(ai_msg.tool_calls)
print("="*50)


####  Groq 사용 가능 모델
- meta-llama/llama-prompt-guard-2-22m
- openai/gpt-oss-120b
- meta-llama/llama-prompt-guard-2-86m
- whisper-large-v3-turbo
- openai/gpt-oss-20b
- qwen/qwen3.8-27b
- whisper-large-v3
- allam-2-7b
- canopylabs/orpheus-v1-english
- openai/gpt-oss-safeguard-20b
- canopylabs/orpheus-arabic-saudi

In [ ]:
# initialize groq modelq
groq_model = ChatGroq(model="openai/gpt-oss-20b", temperature=0.0, max_retries = 2, reasoning_format = "parsed")

# initialize web search tool
web_search = TavilySearch(max_results=2)

# binding tool with llm model
groq_with_tools = groq_model.bind_tools(tools = [web_search])

In [ ]:
# case : tool calling required 
query = "오늘의 최신 뉴스 검색해줘."
ai_msg = groq_with_tools.invoke(query)

# print results
pprint(ai_msg)
print("="*50)

# print content of ai_msg
pprint(ai_msg.content)
print("="*50)

# print tool calls
pprint(ai_msg.tool_calls)
print("="*50)

In [ ]:
tool_call = ai_msg.tool_calls[0]
tool_call

(3) tool implementation

In [ ]:
### 방법 1: 직접 도구 호출 처리

# 이 방법은 AI 메시지에서 첫 번째 도구 호출을 가져와 직접 처리한다.
# 'args'를 사용하여 도구를 호출하고 결과를 얻는다.

tool_output = web_search.invoke(tool_call["args"])
print(f"{tool_call['name']} 호출 결과:")
print("-" * 100)
print(tool_output)

In [ ]:
### 방법 2: ToolMessage 객체 생성

# 이 방법은 도구 호출 결과를 사용하여 ToolMessage 객체를 생성한다.
# 도구 호출의 ID와 이름을 포함하여 더 구조화된 메시지를 만든다.

from langchain_core.messages import ToolMessage
tool_message = ToolMessage(
    content=tool_output,
    tool_call_id=tool_call["id"],
    name=tool_call["name"]
)

print(tool_message)

In [ ]:
### 방법 3: 도구 직접 호출하여 바로 ToolMessage 객체 생성

# 이 방법은 도구를 직접 호출하여 ToolMessage 객체를 생성한다.
# 가장 간단하고 직관적인 방법으로, LangChain의 추상화를 활용한다.

tool_message = web_search.invoke(tool_call)

print(tool_message)


In [ ]:
pprint(tool_message.tool_call_id)
pprint(tool_message.name)
pprint(tool_message.content)

In [ ]:
ai_msg.tool_calls

In [ ]:
# batch 실행 - 도구 호출이 여러 개인 경우

# tool_messages = web_search.batch([tool_call])

tool_messages = web_search.batch(ai_msg.tool_calls)

print(tool_messages)
print("-" * 100)
pprint(tool_messages[0].content)

(4) ToolMessage를 LLM에 전달하여 답변을 생성하기

In [ ]:
from datetime import datetime
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableConfig, chain

# write prompt template
today = datetime.today().strftime("%Y-%m-%d")

prompt = ChatPromptTemplate([
    ("system", f"You are a helpful assistant that can answer questions and help with tasks. Today's date is {today}."),
    ("human", "{user_input}"),
    ("placeholder", "{messages}"),
])

# initialize groq modelq
groq_model = ChatGroq(model="openai/gpt-oss-20b", temperature=0.0, max_retries = 2, reasoning_format = "parsed")

# binding tool with llm model
groq_with_tools = groq_model.bind_tools(tools = [web_search])

# generate LLM chain
# LLM 체인 생성 : prompt의 output -> llm_with_tools의 input 이라는 의미
# 즉, formatted_prompt = prompt.invoke(input_) -> result = groq_with_tools.invoke(formatted_prompt)
groq_chain = prompt | groq_with_tools 

# tool calling chain definition
# 도구 실행 체인 정의
# @chain : 일반 파이썬함수를 runnable 하게 바꿔주는 decorator 
# Runnable: langchain에서 "입력을 받아 출력을 반환하는 실행 가능한 구성 요소" : .invoke(), .batch(), .stream() 등을 사용할 수 있는 객체
@chain
def web_search_chain(user_input: str, config: RunnableConfig):
    # 1. 사용자 질문 준비   
    input_ = {"user_input": user_input}

    # 2. LLM 에게 질문
    ai_msg = groq_chain.invoke(input_, config = config)
    print("ai_msg: \n", ai_msg)
    print("="*50)

    # 3. LLM이 요청한 tool 실행
    tool_msgs = web_search.batch(ai_msg.tool_calls, config = config)
    print("="*50)

    # 4. LLM에게 질문과 tool 결과를 함께 전달 -> LLM이 최종 응답 생성
    return groq_chain.invoke({**input_, "messages": [ai_msg, *tool_msgs]}, config = config)

# execute tool calling chain
response = web_search_chain.invoke("what's the price of bitcoin today?")

# print response
pprint(response.content)


In [ ]:
pprint(response)

현재 코드에는 중요한 한계!! 
- 이 함수는 사실 완전한 Agent는 아님, 왜냐하면 tool call을 딱 한 번만 처리하기 때문에
- 현재 구조: LLM -> Tool -> LLM -> 끝

진짜 Agent는 보통 최종 답변이 나올 때까지 반복 (tool 더 필요하겠는데? 하고 판단할 수 있음)

```
        ┌──────────────┐
        ↓              │
LLM → Tool → LLM → Tool → LLM
        ↑              │
        └──────────────┘
```
- 이걸 관리해주는게 create_agent()


```
@chain
def my_chain(input_data, config: RunnableConfig):

    # 1. input 가공
    x = ...

    # 2. Runnable 실행
    result1 = runnable1.invoke(
        x,
        config=config,
    )

    # 3. 필요하면 다른 Runnable 실행
    result2 = runnable2.invoke(
        result1,
        config=config,
    )

    # 4. 최종 결과 반환
    return result2
```


Runnable
= 실행 가능한 LangChain component
  invoke / batch / stream 가능

@chain
= 내가 만든 Python 함수를 Runnable로 변환

RunnableConfig
= 실제 데이터가 아니라 실행 설정
  tracing / callback / metadata / runtime settings

prompt | model
= Runnable pipeline

bind_tools()
= LLM에게 사용할 수 있는 tool schema 등록

AIMessage.tool_calls
= LLM이 "이 도구를 호출해 주세요"라고 만든 요청

ToolMessage
= 실제 tool 실행 결과

[ai_msg, *tool_msgs]
= LLM의 tool 요청 + tool 결과를 다시 context에 추가

* : 리스트 안의 원소들을 하나씩 꺼내서 펼친다.
** : dictionary 풀기

### 2-2. Customized Tool 사용자 정의 도구
- @tool 데코레이터를 사용해서 사용자 정의 도구를 만들 수 있음

(1) tool definition

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq

from langchain_tavily import TavilySearch

from langchain_core.tools import tool 
from typing import List, Dict
import requests


In [ ]:
# tavily 검색 도구 생성 initialize web search tool
tavily_search = TavilySearch(max_results=2, include_answer=False)

# 검색 실행 결과 저장
search_result = tavily_search.invoke(query)

pprint(search_result)

In [ ]:
# 사용자 정의 도구
@tool
def search_web(query: str) -> str:
    '''
    Search the web for information that does not exist in the database or for the latest information.
    '''

    # tavily 검색 도구 생성 initialize web search tool
    tavily_search = TavilySearch(max_results=2, include_answer=False)

    # 검색 실행 결과 저장
    search_result = tavily_search.invoke(query)
    results = search_result["results"]

    # 검색 결과 포맷팅
    formatted_results = "\n---\n".join([f'<Document href="{result["url"]}">\n{result["content"]}\n</Document>' for result in results])

    if len(formatted_results) >0:
        return formatted_results

    return "No information found on the web."

def blog_search(query: str) -> List[Dict]:
    """
    Send a search request to Naver Blog API.
    """   
    url = "https://openapi.naver.com/v1/search/blog.json"
    headers = {
        "X-Naver-Client-Id": "YOUR_CLIENT_ID",
        "X-Naver-Client-Secret": "YOUR_CLIENT_SECRET"
    }
    params = {"query": query, "display": 10, "start": 1, "sort": "sim"}
    
    response = requests.get(url, headers=headers, params=params)

    if response.status_code == 200:
        return response.json()['items']
    else:
        return []

In [ ]:
print(type(search_web))
print(search_web.name)
print(search_web.description)
print(search_web.args_schema.schema())

In [ ]:
query = "스테이크와 어울리는 와인을 추천해주세요."
search_result = search_web.invoke(query)

print(search_result)

(2) LLM 도구 호출 성능 비교

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq

# 기본 LLM
llm_gemini_flash = ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0)
llm_gemini_pro = ChatGoogleGenerativeAI(model="gemini-2.5-pro", temperature=0)
llm_groq = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

# 도구
tools = [search_web]

gemini_flash_with_tools = llm_gemini_flash.bind_tools(tools)
gemini_pro_with_tools = llm_gemini_pro.bind_tools(tools)
groq_with_tools = llm_groq.bind_tools(tools)

- gemini-2.5-flash

In [ ]:
# 도구 호출이 필요한 LLM 호출을 수행
query = "스테이크와 어울리는 와인을 추천해줘."
ai_msg = gemini_flash_with_tools.invoke(query)

# LLM의 전체 출력 결과 출력
pprint(ai_msg)
print("-" * 100)

# 메시지 content 속성 (텍스트 출력)
pprint(ai_msg.content)
print("-" * 100)

# LLM이 호출한 도구 정보 출력
pprint(ai_msg.tool_calls)
print("-" * 100)

### 2-3. Runnable 객체를 도구로 변환
- 문자열이나 dict 입력을 받는 Runnable을 도구로 변환
- as_tool 메서드를 사용


- Document 클래스: Class for storing a piece of text and associated metadata.
```
from langchain_core.documents import Document

Document(
    self,
    page_content: str,
    **kwargs: Any = {},
)

document = Document(
    page_content="Hello, world!", metadata={"source": "https://example.com"}
)


WikipediaLoader(
  self,
  query: str,
  lang: str = 'en',
  load_max_docs: Optional[int] = 25,
  load_all_available_meta: Optional[bool] = False,
  doc_content_chars_max: Optional[int] = 4000
)

```

| Method | 반환 | 방식 | 언제 유용? |
|---|---|---|---|
| `load()` | `list[Document]` | 동기, **전부 로드 후 반환** | 일반적인 스크립트/실습 |
| `lazy_load()` | `Iterator[Document]` | 동기, **하나씩 필요할 때 로드** | 문서가 많거나 스트리밍 처리 |
| `aload()` | `list[Document]` | **비동기**, 전부 로드 후 반환 | FastAPI/async 파이프라인 |

```
일반 Python 함수
search_wiki(input_data)
        ↓
RunnableLambda
LangChain이 실행할 수 있는 Runnable로 변환
        ↓
.as_tool()
LangChain Tool로 변환
        ↑
WikiSearchSchema
Tool이 받을 argument의 구조/타입 정의
        ↓
Agent/LLM이 wiki_search를 호출 가능
```

In [ ]:
from langchain_community.document_loaders import WikipediaLoader
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda
from pydantic import BaseModel, Field
from typing import List

# WikipediaLoader를 사용하여 위키피디아 문서를 검색하는 함수
def search_wiki(input_data: dict) -> List[Document]:
    """Search Wikipedia documents based on user input (query) and return k documents"""
    query = input_data["query"]
    k = input_data.get("k", 2)
    wiki_loader = WikipediaLoader(query = query, load_max_docs = k, lang = "ko")
    
    # Load data into Document objects.
    wiki_docs = wiki_loader.load()
    return wiki_docs

# 도구 호출에 사용할 입력스키마 정의
class WikiSearchSchema(BaseModel):
    """Input schema for Wikipidea search. """
    query: str = Field(..., description="The query to search Wikipedia for.")
    k: int = Field(2, description = "The number of documents to return (default: 2)")


# RunnableLambda를 사용하여 도구 정의
runnable = RunnableLambda[dict, List[Document]](search_wiki)
wiki_search = runnable.as_tool(
    name = "wiki_search",
    description = dedent("""
        Use this tool when you need to search for information on Wikipedia.
        It searches for Wikipedia articles related to the user's query and returns
        a specified number of documents. This tool is useful when general knowledge
        or background information is required.
    """),
    args_schema = WikiSearchSchema
)
    


In [ ]:
# 도구 속성
print("자료형: ")
print(type(wiki_search))
print("-"*100)

print("name: ")
print(wiki_search.name)
print("-"*100)

print("description: ")
pprint(wiki_search.description)
print("-"*100)

print("schema: ")
pprint(wiki_search.args_schema.schema())
print("-"*100)

In [ ]:
llm_gemini_flash = ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0)

# binding tools with llm model
gemini_with_tools = llm_gemini_flash.bind_tools(tools = [search_web, wiki_search])

# 검색 실행
query = "부산의 유명한 파스타 맛집은 어디인가요? 그리고 파스타의 유래를 알려주세요."
ai_msg = gemini_with_tools.invoke(query)

# LLM의 전체 출력 결과 출력
pprint(ai_msg)
print("-" * 100)

pprint(ai_msg.content)
print("-" * 100)

pprint(ai_msg.tool_calls)
print("-" * 100)



(2) LCEL 체인 (LangChain Expression Language)
- 여러 LangChain component를 데이터가 흐르는 pipeline으로 조립하는 문법
- 예제: 위키피디아 문서를 검색하고 내용을 요약하는 체인

In [ ]:
import wikipedia

wikipedia.set_user_agent(
    "MyLangChainWikipediaApp/1.0"
)

from langchain_community.document_loaders import WikipediaLoader

wiki_loader = WikipediaLoader(
    query="파스타의 유래",
    load_max_docs=2,
    lang="ko"
)

wiki_docs = wiki_loader.load()

pprint(wiki_docs)

```
                  ┌─ Document Loader
                  │
WikipediaLoader ──┤
                  └─ 외부 데이터를 Document로 가져옴
                           ↓
                    List[Document]
                           ↓
RunnableLambda ─── 일반 Python 함수를
                  LangChain Runnable로 변환
                           ↓
ChatPromptTemplate ── LLM에게 전달할 prompt 구성
                           ↓
                         LLM
                           ↓
                       AIMessage
                           ↓
StrOutputParser ─── 필요한 문자열만 추출
                           ↓
                          str
```

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_community.document_loaders import WikipediaLoader

# WikipediaLoader를 사용하여 위키피디아 문서를 검색하고 텍스트로 반환하는 함수 
def wiki_search_to_text(input_data: dict):
    wiki_loader = WikipediaLoader(query = input_data["query"], load_max_docs = 2, lang = "ko")
    wiki_docs = wiki_loader.load()

    # Document를 LLM이 읽기 쉬운 텍스트로 변환
    formatted_docs = [ 
        f"<Document source = '{doc.metadata['source']}'>\n{doc.page_content}\n</Document>" 
        for doc in wiki_docs
        ]

    return formatted_docs

# summary prompt template : LLM에 전달한 메시지를 만듬
summary_prompt = ChatPromptTemplate.from_template(
    "Summarize the following text in a concise manner:\n\n{context}\n\nSummary:"
)

# Set llm summary chain
# llm_gemini_flash 을 거치면 AIMessage 객체로 반환 -> 최종적으로 요약된 문자열만 받고 싶다. 
# -> StrOutputParser() 사용 (LLM 응답에서 최종 text를 꺼내 문자열 형태로 반환)
summary_chain = (
    {"context": RunnableLambda(wiki_search_to_text)} 
    | summary_prompt | llm_gemini_flash | StrOutputParser()
)

# 요약 테스트
summarized_text = summary_chain.invoke({"query":"파스타의 유래"})

pprint(summarized_text)


In [ ]:
# LCEL 를 tool 로 변환

# 도구 호출에 사용할 입력 스키마 정의 
class WikiSearchSchema(BaseModel):
    """Input schema for Wikipidea search. """
    query: str = Field(..., description = "The query to search Wikipedia for.")


# as_tool 메서드를 사용하여 RunnableLambda를 tool로 변환
wiki_summary = summary_chain.as_tool(
    name = "wiki_summary",
    description = dedent("""
        Search Wikipedia for the user's query and return a concise summary.
        Use this tool for general knowledge or background information.
    """).strip(),
    args_schema = WikiSearchSchema
)


# 도구 속성
print("자료형: ")
print(type(wiki_summary))
print("="*50)

print("name: ")
print(wiki_summary.name)
print("="*50)

print("description: ")
pprint(wiki_summary.description)
print("="*50)

print("schema: ")
pprint(wiki_summary.args_schema.schema())
print("="*50)


In [ ]:
# binding tools with llm
gemini_with_tools = llm_gemini_flash.bind_tools(tools = [search_web, wiki_summary])

# 검색 실행
query = "부산의 유명한 파스타 맛집은 어디인가요? 그리고 위키피디아에서 파스타의 유래를 알려줘."
ai_msg = gemini_with_tools.invoke(query)

# LLM의 전체 출력 결과 출력
pprint(ai_msg)
print("-" * 100)

# 메시지 content 속성 (텍스트 출력)
pprint(ai_msg.content)
print("-" * 100)

# LLM이 호출한 도구 정보 출력
pprint(ai_msg.tool_calls)
print("-" * 100)

In [ ]:
# 도구 실행
tool_message = wiki_summary.invoke(ai_msg.tool_calls[0])

print(tool_message)
print("="*50)
pprint(tool_message.content)

```
① 단순 LCEL pipeline

Wikipedia
   ↓
Prompt
   ↓
LLM
   ↓
Parser

개발자가 실행 순서를 완전히 결정
A | B | C


② @chain custom workflow

       LLM
        ↓
   ToolCall 확인
        ↓
    Tool 실행
        ↓
       LLM

개발자가 Python 코드로
실행 흐름을 직접 제어

③ Agent

             ┌─────────────┐
             ↓             │
User → LLM → 판단          │
        │      │            │
        │    Tool 필요      │
        │      ↓            │
        │    Tool ──────────┘
        │
        └─ Tool 불필요
             ↓
          Final Answer
```

In [ ]:
from datetime import datetime
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableConfig, chain

# 현재 시간 가져오기
today = datetime.today().strftime("%Y-%m-%d")

# 프롬프트 템플릿 생성
prompt = ChatPromptTemplate([
    ("system", f"You are a helpful AI assistant. Today's date is {today}."),
    ("human", "{user_input}"),
    ("placeholder", "{messages}"),
])

# LLM에 도구 바인딩
gemini_summary_with_tools = llm_gemini_flash.bind_tools(tools = [wiki_summary])

# langchain
llm_chain = prompt | gemini_summary_with_tools

# 도구 실행 체인 정의
@chain
def wiki_summary_chain(user_input: str, config: RunnableConfig):
    input_ = {"user_input": user_input}
    
    # ① LLM 호출
    ai_msg = llm_chain.invoke(input_, config=config)
    print("ai_msg: \n", ai_msg)
    print("-"*100)

    if not ai_msg.tool_calls:
        return ai_msg

    # ② LLM이 요청한 tool call 확인
    tool_msgs = wiki_summary.batch(ai_msg.tool_calls, config=config)
    print("tool_msgs: \n", tool_msgs)
    print("-"*100)
    
    # ③ Tool 결과를 다시 LLM에게 전달
    return llm_chain.invoke({**input_, "messages": [ai_msg, *tool_msgs]}, config=config)

# 체인 실행
response = wiki_summary_chain.invoke("파스타의 유래에 대해서 알려주세요.")

# 응답 출력 
pprint(response.content)

```
bind_tools(tools)
       │
       │ Tool들의 schema를
       │ LLM에게 알려줌
       ↓
      LLM
       │
       │ "search_web 쓰겠다"
       ↓
 ToolCall
 name="search_web"
       │
       ↓
tools_by_name["search_web"]
       │
       ↓
search_web.invoke(...)
```

In [ ]:
from datetime import datetime
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableConfig, chain

# 현재 시간 가져오기
today = datetime.today().strftime("%Y-%m-%d")

# 프롬프트 템플릿 생성
prompt = ChatPromptTemplate([
    ("system", f"You are a helpful AI assistant. Today's date is {today}."),
    ("human", "{user_input}"),
    ("placeholder", "{messages}"),
])

# LLM에 도구 바인딩
tools = [wiki_summary, search_web]
tools_by_name = {tool.name: tool for tool in tools}
gemini_flash_with_tools = llm_gemini_flash.bind_tools(tools = tools)


# langchain
llm_chain = prompt | gemini_flash_with_tools


@chain
def wiki_summary_chain(user_input: str, config: RunnableConfig):
    input_ = {"user_input": user_input}
    messages = []
    
    # ① 첫번째 LLM 호출
    ai_msg = llm_chain.invoke({**input_, "messages": messages}, config=config)

    while ai_msg.tool_calls:
        # AI의 tool request를 history에 추가
        messages.append(ai_msg)

        # LLM이 요청한 tool call 확인
        for tool_call in ai_msg.tool_calls:
            selected_tool = tools_by_name[tool_call["name"]]

            tool_msg = selected_tool.invoke(tool_call, config = config)
        
            # Tool 결과도 history에 추가
            messages.append(tool_msg)

        # ③ 현재까지의 전체 history 를 가지고 LLM 다시 호출
        ai_msg = llm_chain.invoke({**input_, "messages": messages}, config=config)

    return ai_msg   

# 체인 실행
response = wiki_summary_chain.invoke("파스타의 유래에 대해서 알려주세요.")

# 응답 출력 
pprint(response.content)

**RAG에서 왜 embedding이 필요한지**

Chat model과 embedding model은 둘 다 텍스트를 다루지만 **목적 자체가 달라.**

### Chat model은 "텍스트를 만들어내는 모델"

예를 들어 Llama, GPT, Gemini 같은 chat/generative model에:

```text
매콤한 파스타를 추천해줘.
```

라고 주면 결과가:

```text
매콤한 맛을 원하신다면 아라비아따 파스타를 추천합니다.
```

처럼 **새로운 텍스트**야.

즉 대략:

```text
text
 ↓
Chat Model
 ↓
text
```

질문에 답하고, 요약하고, 추론하고, tool을 선택하는 게 주 역할이지.

---

### Embedding model은 "텍스트를 숫자로 바꾸는 모델"

Embedding model에:

```text
"매콤한 토마토 파스타"
```

를 넣으면 답변을 작성하지 않아.

대신 이런 숫자 배열을 반환해:

```python
[
    0.0182,
    -0.2941,
    0.0371,
    0.4821,
    ...
]
```

이걸 **embedding vector**라고 해.

```text
"매콤한 토마토 파스타"
          ↓
   Embedding Model
          ↓
[0.018, -0.294, 0.037, ...]
```

실제 embedding은 수백~수천 개 정도의 숫자로 이루어진 고차원 vector일 수 있어.

중요한 건 이 숫자들이 텍스트의 **의미(semantic information)를 표현하도록 학습되어 있다는 것**이야.

예를 들어 개념적으로:

```text
"매콤한 파스타"
        ↓
     [vector A]

"매운 토마토 스파게티"
        ↓
     [vector B]

"달콤한 초콜릿 케이크"
        ↓
     [vector C]
```

라면 A와 B는 의미가 비슷하니까 vector 공간에서도 가깝게 위치하도록 만드는 거야.

```text
Embedding Space

       매콤한 파스타 ●
                    ● 매운 토마토 스파게티


                                      ● 초콜릿 케이크
```

그래서 컴퓨터가 **"이 두 문장의 의미가 얼마나 비슷하지?"**를 숫자로 계산할 수 있게 돼.

---

## 그래서 네 Chroma에서 embedding이 필요한 거야

지금 네가 만든 `menu_docs`가 예를 들어:

```python
Document(
    page_content="""
    Spicy Arrabbiata Pasta
    Tomato sauce, garlic, chili pepper...
    """
)
```

라고 해보자.

이걸 Chroma에 저장할 때:

```python
menu_db = Chroma.from_documents(
    documents=menu_docs,
    embedding=embeddings_model,
)
```

실제로는 개념적으로 이런 일이 일어나.

```text
menu_docs

"Spicy Arrabbiata Pasta..."
            ↓
     Embedding Model
            ↓
 [0.12, -0.34, 0.81, ...]
            ↓
          Chroma
```

Chroma는 대략:

```text
Document                         Vector

Arrabbiata...        →    [0.12, -0.34, 0.81, ...]
Carbonara...         →    [0.72,  0.11, 0.14, ...]
Truffle Pasta...     →    [0.51, -0.27, 0.39, ...]
Seafood Pasta...     →    [0.31,  0.62, 0.17, ...]
```

이런 식으로 vector를 저장해 놓는 거야.

---

## 그런데 사용자가 한국어로 검색하면?

예를 들어:

```python
menu_db.similarity_search(
    "매콤한 토마토 파스타",
    k=3
)
```

하면 **검색 문장도 같은 embedding model에 넣어.**

```text
"매콤한 토마토 파스타"
          ↓
   EmbeddingGemma
          ↓
[0.14, -0.31, 0.79, ...]
```

그리고 Chroma가 DB에 저장되어 있는 vector들과 비교해.

```text
Query vector
[0.14, -0.31, 0.79, ...]
       │
       │ similarity
       ↓

Arrabbiata   [0.12, -0.34, 0.81, ...]   ← 매우 가까움 ★★★
Carbonara    [0.72,  0.11, 0.14, ...]   ← 멂
Truffle      [0.51, -0.27, 0.39, ...]   ← 중간
```

그래서 가장 가까운 Document들을 찾아오는 거야.

이게 **semantic search / vector search**의 핵심이야.

---

## 그럼 왜 ChatGPT 같은 Chat Model로 이걸 안 하냐?

여기가 핵심 질문이야.

Chat model의 목적은:

```text
input text
    ↓
LLM
    ↓
다음 token 예측
    ↓
문장 생성
```

이고,

Embedding model의 목적은:

```text
input text
    ↓
Embedding Model
    ↓
의미를 나타내는 고정 길이 vector
```

야.

즉 **학습 목적과 출력이 다르다**고 생각하면 돼.

그리고 검색해야 할 문서가 100만 개라고 생각해봐.

매번 질문할 때:

```text
Chat Model에게

문서 1과 질문 비교해주세요.
문서 2와 질문 비교해주세요.
문서 3과 질문 비교해주세요.
...
문서 1,000,000과 질문 비교해주세요.
```

하면 엄청 느리고 비싸겠지.

Embedding을 사용하면 문서는 **미리 한 번 vector로 만들어 놓을 수 있어.**

```text
             최초 한 번

1,000,000 documents
        ↓
Embedding Model
        ↓
1,000,000 vectors
        ↓
Vector DB
```

그리고 사용자가 질문할 때마다:

```text
새로운 질문
    ↓
Embedding Model        ← 이것만 새로 계산
    ↓
query vector
    ↓
Vector DB 검색
    ↓
가장 가까운 10개
```

만 하면 돼.

그래서 빠르고 비용도 훨씬 작아.

---

## 그러면 Chat Model은 언제 등장하냐?

**검색 다음 단계**에서 등장해.

이게 바로 RAG야.

네 메뉴 프로젝트를 전체적으로 보면:

```text
① 메뉴 저장

menu_docs
    ↓
EmbeddingGemma
    ↓
vectors
    ↓
Chroma
```

사용자가:

> "매콤한데 해산물은 없는 파스타 추천해줘"

라고 질문하면:

```text
② Retrieval

사용자 질문
    ↓
EmbeddingGemma
    ↓
query vector
    ↓
Chroma
    ↓
관련 메뉴 3개 검색
```

그 다음에야:

```text
③ Generation

사용자 질문
        +
검색된 메뉴 3개
        ↓
    Chat Model
        ↓
"아라비아따를 추천합니다.
 해산물이 없으며..."
```

가 되는 거야.

즉 RAG에서 역할을 아주 간단히 나누면:

```text
Embedding Model
      ↓
"어떤 문서를 가져올까?"
      ↓
  Retrieval


Chat Model
      ↓
"가져온 문서를 가지고
 어떻게 답할까?"
      ↓
  Generation
```

그래서 **RAG = Retrieval + Augmented Generation**인 거고.

네가 지금 만드는 코드에서 보면:

```text
TextLoader
   ↓
Document
   ↓
메뉴 단위 parsing
   ↓
menu_docs
   ↓
EmbeddingGemma        ← 지금 여기
   ↓
Chroma
   ↓
Retriever
   ↓
Prompt
   ↓
Chat Model
   ↓
Answer
```

이 흐름을 이해해두는 게 중요해. 앞으로 Clinical Trial Navigator 같은 프로젝트를 보면 구조가 똑같이 보이기 시작할 거야.

### 2-4. Vector Store Retriever 
- use '@tool' decorator

(1) load documents and indexing

In [ ]:
from langchain_community.document_loaders import TextLoader

# load menu data from /data/menu.txt
loader = TextLoader("/Volumes/Samsung_9100Pro_2TB/Projects/agent/data/restaurant_menu.txt", encoding = "utf-8")
doc = loader.load()

pprint(doc)

In [ ]:
wine_loader = TextLoader("/Volumes/Samsung_9100Pro_2TB/Projects/agent/data/restaurant_wine.txt", encoding = "utf-8")
wine_doc = wine_loader.load()
pprint(wine_doc)


In [ ]:
import re
from pathlib import Path
from langchain_core.documents import Document


# category classification
def determine_category(menu_name: str, text: str) -> str:
    """determine category of menu by menu name and text."""
    combined = (menu_name + " " + text).lower()
    
    if any(k in combined for k in ["스테이크", "콘피"]):
        return "Main / Meat"
    elif any(k in combined for k in ["리조또", "파스타"]):
        return "Main / Pasta & Rice"
    elif any(k in combined for k in ["수프", "비스크"]):
        return "Soup"
    elif "샐러드" in combined or "타르타르" in combined:
        return "Appetizer / Salad"
    elif any(k in combined for k in ["티라미수", "케이크", "디저트"]):
        return "Dessert"
    return "Other"


# chunking 문서 분할
def split_menu_items(doc: Document) -> list[Document]:

    """하나의 메뉴 Document를 메뉴 항목별 Document로 분리한다."""

    # source를 상대경로로 변환
    source = Path(doc.metadata.get("source", ""))
    project_root = Path.cwd().parent
    try:
        relative_source = str(source.relative_to(project_root))
    except ValueError:
        relative_source = str(source)
    
    text = doc.page_content.strip()

    # split menu items by double newline
    menu_items = re.split(
        r"\n\n(?=\d+\.\s)",
        text
    )

    menu_docs = []

    for item in menu_items:
        item_str = item.strip()
        if not item_str:
            continue

        lines = item_str.split("\n")
        first_line = lines[0].strip()

        # 1. 메뉴 번호 & 메뉴명 추출
        match = re.match(r"^(\d+)\.\s*(.+)", first_line)
        if match:
            menu_number = int(match.group(1))
            menu_name = match.group(2).strip()
        else:
            menu_number = None
            menu_name = first_line

        # 2. 가격(Price) 추출: 예) ₩35,000 -> 35000 (숫자형) 또는 "₩35,000" (문자열형)
        price_match = re.search(r"가격:\s*₩?\s*([\d,]+)", item_str)
        if price_match:
            # 쉼표를 제거하고 정수형(int)으로 저장하면 검색/필터링에 유리합니다.
            price = int(price_match.group(1).replace(",", ""))
        else:
            price = None

        # 3. 주요 식재료(Ingredients) 추출
        ingredients_match = re.search(r"주요 식재료:\s*(.+)", item_str)
        if ingredients_match:
            # 쉼표 구분자로 리스트 변환 (예: ['최상급 한우 등심', '로즈메리 감자', '그릴드 아스파라거스'])
            ingredients_raw = ingredients_match.group(1).strip()
            ingredients = [ing.strip() for ing in ingredients_raw.split(",")]
        else:
            ingredients = []

        # 4. 카테고리(Category) 자동 분류
        category = determine_category(menu_name, item_str)

        # 5. 메타데이터 구성 및 Document 생성
        menu_doc = Document(
            page_content=item_str,
            metadata={
                "source": relative_source,
                "menu_number": menu_number,
                "menu_name": menu_name,
                "price": price,
                "ingredients": ingredients,
                "category": category,
            }
        )

        menu_docs.append(menu_doc)

    return menu_docs

def determine_wine_category(wine_name: str, grapes: list[str], text: str) -> str:
    """와인 이름, 품종, 설명 키워드를 기반으로 와인 종류(Red/White/Sparkling/Dessert)를 분류한다."""
    combined = (wine_name + " " + " ".join(grapes) + " " + text).lower()

    if any(k in combined for k in ["디저트 와인", "소테른", "샤토 디켐", "꿀"]):
        return "Dessert Wine"
    elif any(k in combined for k in ["샴페인", "버블", "프레스티지 큐베", "스파클링"]):
        return "Sparkling Wine"
    elif any(
        k in combined
        for k in ["소비뇽 블랑", "샤르도네", "화이트 와인", "푸이 퓌세", "몽라쉐"]
    ):
        return "White Wine"
    else:
        # 기본적으로 레드 와인 품종(카베르네 소비뇽, 메를로, 네비올로, 시라 등)이 많으므로 Red Wine으로 지정
        return "Red Wine"

def split_wine_items(doc: Document) -> list[Document]:
    """하나의 와인 Document를 와인 항목별 Document로 분리하고 메타데이터를 추출한다."""

    # source 경로 처리
    source_path = Path(doc.metadata.get("source", ""))
    project_root = Path.cwd().parent
    try:
        relative_source = str(source_path.relative_to(project_root))
    except ValueError:
        relative_source = str(source_path)

    text = doc.page_content.strip()

    # 두 개 이상의 줄바꿈 + "숫자." 패턴으로 와인 항목 분할
    wine_items = re.split(r"\n\n+(?=\d+\.\s)", text)

    wine_docs = []

    for item in wine_items:
        item_str = item.strip()
        if not item_str:
            continue

        lines = item_str.split("\n")
        first_line = lines[0].strip()

        # 1. 와인 번호, 와인 전체 이름, 빈티지(생산년도) 추출
        # 예: "1. 샤토 마고 2015" -> number: 1, full_name: "샤토 마고 2015"
        match = re.match(r"^(\d+)\.\s*(.+)", first_line)
        if match:
            wine_number = int(match.group(1))
            full_name = match.group(2).strip()
        else:
            wine_number = None
            full_name = first_line

        # 이름 끝에 붙은 4자리 빈티지 연도 추출 (예: 2015)
        vintage_match = re.search(r"\b(19\d{2}|20\d{2})\b", full_name)
        if vintage_match:
            vintage = int(vintage_match.group(1))
            # 빈티지를 제외한 순수 와인 이름
            wine_name = full_name.replace(str(vintage), "").strip()
        else:
            vintage = None
            wine_name = full_name

        # 2. 가격 및 용량 추출
        # 예: "₩800,000 (375ml)" 또는 "₩450,000"
        price_match = re.search(r"가격:\s*₩?\s*([\d,]+)", item_str)
        if price_match:
            price = int(price_match.group(1).replace(",", ""))
        else:
            price = None

        # 용량(ml) 추출 (명시되지 않은 경우 standard 750ml 처리)
        vol_match = re.search(r"(\d+)\s*ml", item_str, re.IGNORECASE)
        volume_ml = int(vol_match.group(1)) if vol_match else 750

        # 3. 주요 품종 추출 (쉼표로 구분하여 리스트 변환)
        grapes_match = re.search(r"주요 품종:\s*(.+)", item_str)
        if grapes_match:
            grapes_raw = grapes_match.group(1).strip()
            grapes = [g.strip() for g in grapes_raw.split(",")]
        else:
            grapes = []

        # 4. 와인 카테고리 추출
        category = determine_wine_category(wine_name, grapes, item_str)

        # 5. Document 생성
        wine_doc = Document(
            page_content=item_str,
            metadata={
                "source": relative_source,
                "wine_number": wine_number,
                "wine_name": wine_name,
                "vintage": vintage,
                "price": price,
                "volume_ml": volume_ml,
                "grapes": grapes,
                "category": category,
            },
        )

        wine_docs.append(wine_doc)

    return wine_docs    


In [ ]:
menu_docs = split_menu_items(doc[0])
wine_docs = split_wine_items(wine_doc[0])

In [ ]:
pprint(menu_docs)

In [ ]:
for menu in menu_docs:
    print(f"메뉴 번호: {menu.metadata['menu_number']}")
    print(f"메뉴 이름: {menu.metadata['menu_name']}")
    print("-"*50)

from langchain_text_splitters import RecursiveCharacterTextSplitter



In [ ]:
import shutil
# Chroma Vectorstore를 사용하기 위한 준비
from langchain_chroma import Chroma
from langchain_ollama  import OllamaEmbeddings
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
CHROMA_DIR = PROJECT_ROOT / "chroma_db"
DATA_DIR = PROJECT_ROOT / "data"

# DB를 지우고 새로 생성시
# if CHROMA_DIR.exists():
#     shutil.rmtree(CHROMA_DIR)

In [ ]:
embeddings_model = OllamaEmbeddings(model="qwen3-embedding:0.6b") 

# Chroma 인덱스 생성
# Embedding model은 page_content를 vector로 변환하고, 
# 따라서 (검색에 의미적으로 중요한 정보라면 metadata에만 넣지 말고 page_content에도 포함시키는 것이 중요한 설계!)
# Chroma는 원본 document + vector + metadata를 함께 관리한다.
menu_db = Chroma.from_documents(
    documents = menu_docs,
    embedding = embeddings_model,
    collection_name = "restaurant_menu",
    persist_directory = str(CHROMA_DIR),
)

wine_db = Chroma.from_documents(
    documents = wine_docs,
    embedding = embeddings_model,
    collection_name = "restaurant_wine",
    persist_directory = str(CHROMA_DIR),
)

# Retriever 생성
# Retriever는 쿼리에 대한 검색(Chroma가 vectro similarity로 가장 가까운 record를 찾음)을 하고 Document를 반환
# RAG(Retrieval + Augmented Generation)는 Retrieved 된 데이터를 가지고 chat model에게 전달하여 답변을 생성
menu_retriever = menu_db.as_retriever(
    search_kwargs = {"k": 2},
)

# 쿼리 테스트
query = "시그니처 스테이크 설명해줘."
docs = menu_retriever.invoke(query)
print(f"검색 결과: {len(docs)}개의 문서")

for doc in docs:
    print(f"메뉴 번호: {doc.metadata['menu_number']}")
    print(f"메뉴 이름: {doc.metadata['menu_name']}")
    print("-"*50)
    

In [ ]:
test_vector = embeddings_model.embed_query("테스트")
print(len(test_vector))

(2) tool definition

In [ ]:
from typing import List
from langchain_core.tools import tool

# 1. load vector store
embeddings_model = OllamaEmbeddings(model="qwen3-embedding:0.6b") 

menu_db = Chroma.from_documents(
    documents = menu_docs,
    embedding = embeddings_model,
    collection_name = "restaurant_menu",
    persist_directory = str(CHROMA_DIR),
)


# 2. define tool
@tool
def search_menu(query: str) -> List[Document]:
    """
    Securely retrieve and access authorized restaurant menu information from the encrypted database.
    Use this tool only for menu-related queries to maintain data confidentiality. 
    """
    docs = menu_db.similarity_search(query, k = 3)
    if len(docs) > 0:
        return docs
    else:
        return [Document(page_content = "Cannot find any related menu information.")]


@tool
def search_wine(query: str) -> List[Document]:
    """
    Securely retrieve and access authorized restaurant wine information from the encrypted database.
    Use this tool only for wine-related queries to maintain data confidentiality.
    """
    docs = wine_db.similarity_search(query, k = 3)
    if len(docs) > 0:
        return docs
    else:
        return [Document(page_content = "Cannot find any related wine information.")]


In [ ]:
# 도구 속성
print("자료형: ")
print(type(search_wine))
print("-"*100)

print("schema: ")
pprint(search_wine.args_schema.schema())
print("-"*100)

In [ ]:
# initialize web search tool
web_search = TavilySearch(max_results=2)

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq

# 기본 LLM
llm_gemini_flash = ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0)
llm_groq = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

# tools
tools = [search_menu, search_wine]

# binding tools with llm model
gemini_flash_with_tools = llm_gemini_flash.bind_tools(tools = tools)
groq_with_tools = llm_groq.bind_tools(tools = tools)

# 쿼리 테스트
query = "해산물 메뉴 추천해주고, 그 메뉴와 어울리는 와인도 추천해줘."
ai_msg = gemini_flash_with_tools.invoke(query)




In [ ]:
print(ai_msg.tool_calls)


In [ ]:
from datetime import datetime
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableConfig, chain

tools = [search_menu, search_wine, web_search, wiki_summary]
tools_by_name = {tool.name: tool for tool in tools}

# today
today = datetime.today().strftime("%Y-%m-%d")

# prompt template
prompt = ChatPromptTemplate([
    ("system", f"You are a professional restaurant manager. and your duty is to recommend the best menu and wine for the customer. Today's date is {today}."),
    ("human", "{user_input}"),
    ("placeholder", "{messages}"),
])

# binding tools with llm model
gemini_with_tools = llm_gemini_flash.bind_tools(tools = tools)

# langchain
llm_chain = prompt | gemini_flash_with_tools


@chain
def restaurant_recommendation_chain(user_input: str, config: RunnableConfig):
    messages = []
    
    # ① 첫번째 LLM 호출
    ai_msg = llm_chain.invoke({"user_input": user_input, "messages": messages}, config=config)

    while ai_msg.tool_calls:
        # AI의 tool request를 history에 추가
        messages.append(ai_msg)

        # LLM이 요청한 tool call 확인
        for tool_call in ai_msg.tool_calls:
            print(f"{tool_call['name']}: \n{tool_call}")
            selected_tool = tools_by_name[tool_call["name"]]

            tool_msg = selected_tool.invoke(tool_call, config = config)
        
            # Tool 결과도 history에 추가
            messages.append(tool_msg)

        # ③ 현재까지의 전체 history 를 가지고 LLM 다시 호출
        ai_msg = llm_chain.invoke({"user_input": user_input, "messages": messages}, config=config)


    return ai_msg   

        


In [ ]:
response = restaurant_recommendation_chain.invoke("해산물 메뉴 추천해주고, 그 메뉴와 어울리는 와인도 추천해줘.")
print(response.content)

### 3-1. Few-Shot Prompting or Zero-Shot
##### Few-Shot
- 모델에게 몇가지 예시를 제공하여 원하는 출력형식이나 작업 수행 방식을 보여주는 기법
- 즉, 모델에게 도구를 어떻게 사용해야하 하는지 예시를 통해 보여주기 위함

##### Zero-Shot
- 모델에게 별도의 예시(Example)를 제공하지 않고, 명확한 지시문(Instruction/System Prompt)만으로 원하는 작업이나 도구 사용을 수행하게 하는 기법
- 즉, 모델의 사전 학습된 지식과 지시 이행 능력(Instruction Following)에만 의존하여 정해진 규칙에 따라 도구를 선택하고 호출하게 함

##### apply to Tool Calling 
1. Define System Prompt & Tools: 도구의 세부 명세(description, args_schema)와 도구 사용 규칙/조건을 지시문으로 명확히 정의
2. Make a Prompt Template: System Prompt와 사용자 입력({query})만으로 심플하게 템플릿 구성 (예시 메시지 제외)
3. Generate Prompt & Call Model: LLM이 지시문을 바탕으로 필요 여부를 스스스로 판단하여 tool_calls 결정
4. Parsing Response / Execute Tool: 모델의 요청에 따라 Tool을 실행하고 최종 답변 생성

| 구분 | Zero-Shot Prompting | Few-Shot Prompting |
| :--- | :--- | :--- |
| **예시 제공 여부** | 없음 (System Prompt + Query만 사용) | 있음 (대화/도구 호출 이력 예시 수개 포함) |
| **토큰량 & 비용** | 최소화 (응답 속도가 빠르고 비용 절감) | 증가 (예시 분량만큼 입력 토큰 발생) |
| **주요 의존도** | LLM의 기본 추론 및 지시 이행 능력 | 예시를 통한 패턴 학습 및 출력 포맷 고정 |
| **추천 상황** | 최신 고성능 모델(Gemini, GPT-4o 등) 사용 시, 단일/단순 도구 호출 | 복잡한 Multi-step Tool Calling, 특정 도구 호출 순서/포맷 강제가 필요한 경우 |

In [ ]:
from langchain_core.prompts import ChatPromptTemplate


system_prompt = """You are a helpful AI assistant for a restaurant. Your job is to answer user queries accurately using the provided tools.

### Guidelines for Tool Usage:
1. **search_menu**: Use ONLY when looking up specific restaurant menu details (prices, ingredients, descriptions).
2. **search_wine**: Use ONLY when the user asks for wine recommendations, wine details, or food pairings.
3. **wiki_summary**: Use ONLY for general culinary/food knowledge not covered by the restaurant menu.
4. **search_web**: Use ONLY for real-time information or when internal tools don't yield results.

### Rules:
- **Be Selective**: Do NOT call tools that are irrelevant to the user's explicit question. (e.g., If the user asks only about a menu price, do NOT call `search_wine` or `wiki_summary`).
- **Chain Tools Wisely**: If a query asks for both menu info and matching wine, call `search_menu` first, then call `search_wine`.
- **Accuracy**: Base your answer STRICTLY on the tool outputs. Do not invent prices or ingredients.
"""

# Few-shot(examples)을 빼고 간결한 Zero-shot 구조로 변경
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{query}"),
        {"placeholder", "{messages}"},
    ]
)


# ChatOpenAI 모델 초기화 
llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash")

# 검색 도구를 직접 LLM에 바인딩 가능
llm_with_tools = llm.bind_tools(tools=tools)

# zero-shot 프롬프트를 사용한 체인 구성
zero_shot_search_chain = prompt | llm_with_tools


@chain
def restaurant_zero_shot_chain(query: str, config: RunnableConfig):
    messages = []

    # 첫번재 LLM 호출
    ai_msg = zero_shot_search_chain.invoke({"query": query, "messages": messages}, config = config)

    while ai_msg.tool_calls:
        # AI의 tool request를 history에 추가
        messages.append(ai_msg)

        # llm이 요청한 tool call 확인
        for tool_call in ai_msg.tool_calls:
            selected_tool = tools_by_name[tool_call["name"]]
            tool_msg = selected_tool.invoke(tool_call, config = config)

            # Tool 결과도 history에 추가
            messages.append(tool_msg)

        # ③ 현재까지의 전체 history 를 가지고 LLM 다시 호출
        ai_msg = zero_shot_search_chain.invoke({"query": query, "messages": messages}, config = config)

    return ai_msg

# 체인 실행
query = "스테이크 메뉴가 있나요? 스테이크와 어울리는 와인을 추천해주세요."
response = restaurant_zero_shot_chain.invoke(query)

In [ ]:
import re
import os, json

# dedent : 여러줄 문자열을 코드상 보기 좋게 들여쓰기했을 때, 공통 들여쓰기를 제거하는 함수 (불필요한 앞쪽 공백 제거)
from textwrap import dedent
from pprint import pprint

import warnings

warnings.filterwarnings("ignore")

from dotenv import load_dotenv

load_dotenv()

### 3-2. Agent (LangChain 1.x)


#### 핵심 API: `create_agent`

```python
from langchain.agents import create_agent

agent = create_agent(
    model=llm,                    # ChatModel 인스턴스 또는 "google_genai:gemini-2.5-flash" 같은 문자열
    tools=[search_menu, ...],     # @tool / BaseTool 목록
    system_prompt="...",          # 시스템 지시문 (문자열 또는 SystemMessage)
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "질문"}]
})
answer = result["messages"][-1].content
```

- **반환 타입**: LangGraph `CompiledStateGraph` → `.invoke()`, `.stream()` 사용
- **동작**: LLM이 tool을 더 호출할 때까지 **자동 반복** (앞에서 `@chain` + `while ai_msg.tool_calls`로 만든 루프와 동일한 역할)
- **대화 기억**(선택): `checkpointer=InMemorySaver()` + `config={"configurable": {"thread_id": "..."}}`

#### 0.x → 1.x 매핑

| 구버전 (LangChain 0.x) | 신버전 (이 프로젝트) |
| :--- | :--- |
| `create_tool_calling_agent(llm, tools, prompt)` | `create_agent(model, tools, system_prompt=...)` |
| `AgentExecutor(agent=..., tools=...)` | **제거** — 실행 루프는 `create_agent`에 내장 |
| `agent_executor.invoke({"input": "...", "chat_history": [...]})` | `agent.invoke({"messages": [...]})` |
| 프롬프트에 `{input}`, `agent_scratchpad` 필수 | `system_prompt` + 사용자 메시지는 `messages` |

#### `langchain.agents`에서 자주 쓰는 것 (1.x)

| Import | 용도 |
| :--- | :--- |
| `create_agent` | **권장** — tool calling 에이전트 생성 |
| `middleware` | HITL, 로깅, 커스텀 훅 등 (고급) |

#### 더 이상 `langchain.agents`에서 쓰지 않는 것 (구버전)

| 구 API | 1.x에서 |
| :--- | :--- |
| `create_tool_calling_agent` | → `create_agent` |
| `create_react_agent` | → LangGraph로 ReAct 그래프 직접 구성, 또는 tool calling 가능 모델 + `create_agent` |
| `create_structured_output_agent` | → `create_agent(..., response_format=YourPydanticModel)` 등 structured output 옵션 |
| `AgentExecutor` | → 불필요 (`create_agent`가 실행) |
| `Tool`, `load_tools` | → `langchain_core.tools.tool` / `@tool`, 통합 패키지·커뮤니티 도구 |

#### Tool · 프롬프트 정리

- **도구 정의**: `from langchain_core.tools import tool` (또는 `langchain.tools`)
- **시스템 프롬프트**: `ChatPromptTemplate` + `MessagesPlaceholder("agent_scratchpad")` 대신 **`system_prompt` 문자열**을 `create_agent`에 전달
- **프롬프트 템플릿**: 단순 `@chain` 체인에서는 여전히 `langchain_core.prompts.ChatPromptTemplate` 사용 가능 (Agent 섹션과는 별개)

복잡한 분기·서브그래프·장기 실행이 필요하면 LangGraph로 그래프를 직접 설계하는 방향이 공식 문서의 다음 단계입니다.

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_tavily import TavilySearch
from langchain_core.tools import tool 
from typing import List
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_community.document_loaders import WikipediaLoader
from langchain_community.document_loaders import TextLoader
import re
from pathlib import Path

from langchain_chroma import Chroma
from langchain_ollama  import OllamaEmbeddings

/var/folders/fh/2dgg5czs11s592c8_k1gbtlc0000gn/T/ipykernel_72715/1371429584.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WikipediaLoader


In [3]:
import re
import os, json

# dedent : 여러줄 문자열을 코드상 보기 좋게 들여쓰기했을 때, 공통 들여쓰기를 제거하는 함수 (불필요한 앞쪽 공백 제거)
from textwrap import dedent
from pprint import pprint

import warnings

warnings.filterwarnings("ignore")

from dotenv import load_dotenv

load_dotenv()

True

In [4]:
llm_gemini_flash = ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0)

In [5]:
@tool
def search_web(query: str) -> str:
    '''
    Search the web for information that does not exist in the database or for the latest information.
    '''

    # tavily 검색 도구 생성 initialize web search tool
    tavily_search = TavilySearch(max_results=2, include_answer=False)

    # 검색 실행 결과 저장
    search_result = tavily_search.invoke(query)
    results = search_result["results"]

    # 검색 결과 포맷팅
    formatted_results = "\n---\n".join([f'<Document href="{result["url"]}">\n{result["content"]}\n</Document>' for result in results])

    if len(formatted_results) >0:
        return formatted_results

    return "No information found on the web."

In [6]:
# wiki_summary tool

# WikipediaLoader를 사용하여 위키피디아 문서를 검색하고 텍스트로 반환하는 함수 
def wiki_search_to_text(input_data: dict):
    wiki_loader = WikipediaLoader(query = input_data["query"], load_max_docs = 2, lang = "ko")
    wiki_docs = wiki_loader.load()

    # Document를 LLM이 읽기 쉬운 텍스트로 변환
    formatted_docs = [ 
        f"<Document source = '{doc.metadata['source']}'>\n{doc.page_content}\n</Document>" 
        for doc in wiki_docs
        ]

    return formatted_docs

# summary prompt template : LLM에 전달한 메시지를 만듬
summary_prompt = ChatPromptTemplate.from_template(
    "Summarize the following text in a concise manner:\n\n{context}\n\nSummary:"
)

# Set llm summary chain
# llm_gemini_flash 을 거치면 AIMessage 객체로 반환 -> 최종적으로 요약된 문자열만 받고 싶다. 
# -> StrOutputParser() 사용 (LLM 응답에서 최종 text를 꺼내 문자열 형태로 반환)
summary_chain = (
    {"context": RunnableLambda(wiki_search_to_text)} 
    | summary_prompt | llm_gemini_flash | StrOutputParser()
)

# 도구 호출에 사용할 입력 스키마 정의 
class WikiSearchSchema(BaseModel):
    """Input schema for Wikipidea search. """
    query: str = Field(..., description = "The query to search Wikipedia for.")


# as_tool 메서드를 사용하여 RunnableLambda를 tool로 변환
wiki_summary = summary_chain.as_tool(
    name = "wiki_summary",
    description = dedent("""
            Search Wikipedia for the user's query and return a concise summary.
            Use this tool for general knowledge or background information.
        """).strip(),
    args_schema = WikiSearchSchema
    )

In [7]:
# load menu data from /data/menu.txt
loader = TextLoader("/Volumes/Samsung_9100Pro_2TB/Projects/agent/data/restaurant_menu.txt", encoding = "utf-8")
doc = loader.load()

pprint(doc)

wine_loader = TextLoader("/Volumes/Samsung_9100Pro_2TB/Projects/agent/data/restaurant_wine.txt", encoding = "utf-8")
wine_doc = wine_loader.load()
pprint(wine_doc)




# category classification
def determine_category(menu_name: str, text: str) -> str:
    """determine category of menu by menu name and text."""
    combined = (menu_name + " " + text).lower()
    if any(k in combined for k in ["스테이크", "콘피"]):
        return "Main / Meat"
    elif any(k in combined for k in ["리조또", "파스타"]):
        return "Main / Pasta & Rice"
    elif any(k in combined for k in ["수프", "비스크"]):
        return "Soup"
    elif "샐러드" in combined or "타르타르" in combined:
        return "Appetizer / Salad"
    elif any(k in combined for k in ["티라미수", "케이크", "디저트"]):
        return "Dessert"
    return "Other"

# chunking 문서 분할
def split_menu_items(doc: Document) -> list[Document]:

    """하나의 메뉴 Document를 메뉴 항목별 Document로 분리한다."""

    # source를 상대경로로 변환
    source = Path(doc.metadata.get("source", ""))
    project_root = Path.cwd().parent
    try:
        relative_source = str(source.relative_to(project_root))
    except ValueError:
        relative_source = str(source)
    
    text = doc.page_content.strip()

    # split menu items by double newline
    menu_items = re.split(
        r"\n\n(?=\d+\.\s)",
        text
    )

    menu_docs = []

    for item in menu_items:
        item_str = item.strip()
        if not item_str:
            continue

        lines = item_str.split("\n")
        first_line = lines[0].strip()

        # 1. 메뉴 번호 & 메뉴명 추출
        match = re.match(r"^(\d+)\.\s*(.+)", first_line)
        if match:
            menu_number = int(match.group(1))
            menu_name = match.group(2).strip()
        else:
            menu_number = None
            menu_name = first_line

        # 2. 가격(Price) 추출: 예) ₩35,000 -> 35000 (숫자형) 또는 "₩35,000" (문자열형)
        price_match = re.search(r"가격:\s*₩?\s*([\d,]+)", item_str)
        if price_match:
            # 쉼표를 제거하고 정수형(int)으로 저장하면 검색/필터링에 유리합니다.
            price = int(price_match.group(1).replace(",", ""))
        else:
            price = None

        # 3. 주요 식재료(Ingredients) 추출
        ingredients_match = re.search(r"주요 식재료:\s*(.+)", item_str)
        if ingredients_match:
            # 쉼표 구분자로 리스트 변환 (예: ['최상급 한우 등심', '로즈메리 감자', '그릴드 아스파라거스'])
            ingredients_raw = ingredients_match.group(1).strip()
            ingredients = [ing.strip() for ing in ingredients_raw.split(",")]
        else:
            ingredients = []

        # 4. 카테고리(Category) 자동 분류
        category = determine_category(menu_name, item_str)

        # 5. 메타데이터 구성 및 Document 생성
        menu_doc = Document(
            page_content=item_str,
            metadata={
                "source": relative_source,
                "menu_number": menu_number,
                "menu_name": menu_name,
                "price": price,
                "ingredients": ingredients,
                "category": category,
            }
        )

        menu_docs.append(menu_doc)

    return menu_docs

def determine_wine_category(wine_name: str, grapes: list[str], text: str) -> str:
    """와인 이름, 품종, 설명 키워드를 기반으로 와인 종류(Red/White/Sparkling/Dessert)를 분류한다."""
    combined = (wine_name + " " + " ".join(grapes) + " " + text).lower()

    if any(k in combined for k in ["디저트 와인", "소테른", "샤토 디켐", "꿀"]):
        return "Dessert Wine"
    elif any(k in combined for k in ["샴페인", "버블", "프레스티지 큐베", "스파클링"]):
        return "Sparkling Wine"
    elif any(
        k in combined
        for k in ["소비뇽 블랑", "샤르도네", "화이트 와인", "푸이 퓌세", "몽라쉐"]
    ):
        return "White Wine"
    else:
        # 기본적으로 레드 와인 품종(카베르네 소비뇽, 메를로, 네비올로, 시라 등)이 많으므로 Red Wine으로 지정
        return "Red Wine"

def split_wine_items(doc: Document) -> list[Document]:
    """하나의 와인 Document를 와인 항목별 Document로 분리하고 메타데이터를 추출한다."""

    # source 경로 처리
    source_path = Path(doc.metadata.get("source", ""))
    project_root = Path.cwd().parent
    try:
        relative_source = str(source_path.relative_to(project_root))
    except ValueError:
        relative_source = str(source_path)

    text = doc.page_content.strip()

    # 두 개 이상의 줄바꿈 + "숫자." 패턴으로 와인 항목 분할
    wine_items = re.split(r"\n\n+(?=\d+\.\s)", text)

    wine_docs = []

    for item in wine_items:
        item_str = item.strip()
        if not item_str:
            continue

        lines = item_str.split("\n")
        first_line = lines[0].strip()

        # 1. 와인 번호, 와인 전체 이름, 빈티지(생산년도) 추출
        # 예: "1. 샤토 마고 2015" -> number: 1, full_name: "샤토 마고 2015"
        match = re.match(r"^(\d+)\.\s*(.+)", first_line)
        if match:
            wine_number = int(match.group(1))
            full_name = match.group(2).strip()
        else:
            wine_number = None
            full_name = first_line

        # 이름 끝에 붙은 4자리 빈티지 연도 추출 (예: 2015)
        vintage_match = re.search(r"\b(19\d{2}|20\d{2})\b", full_name)
        if vintage_match:
            vintage = int(vintage_match.group(1))
            # 빈티지를 제외한 순수 와인 이름
            wine_name = full_name.replace(str(vintage), "").strip()
        else:
            vintage = None
            wine_name = full_name

        # 2. 가격 및 용량 추출
        # 예: "₩800,000 (375ml)" 또는 "₩450,000"
        price_match = re.search(r"가격:\s*₩?\s*([\d,]+)", item_str)
        if price_match:
            price = int(price_match.group(1).replace(",", ""))
        else:
            price = None

        # 용량(ml) 추출 (명시되지 않은 경우 standard 750ml 처리)
        vol_match = re.search(r"(\d+)\s*ml", item_str, re.IGNORECASE)
        volume_ml = int(vol_match.group(1)) if vol_match else 750

        # 3. 주요 품종 추출 (쉼표로 구분하여 리스트 변환)
        grapes_match = re.search(r"주요 품종:\s*(.+)", item_str)
        if grapes_match:
            grapes_raw = grapes_match.group(1).strip()
            grapes = [g.strip() for g in grapes_raw.split(",")]
        else:
            grapes = []

        # 4. 와인 카테고리 추출
        category = determine_wine_category(wine_name, grapes, item_str)

        # 5. Document 생성
        wine_doc = Document(
            page_content=item_str,
            metadata={
                "source": relative_source,
                "wine_number": wine_number,
                "wine_name": wine_name,
                "vintage": vintage,
                "price": price,
                "volume_ml": volume_ml,
                "grapes": grapes,
                "category": category,
            },
        )

        wine_docs.append(wine_doc)

    return wine_docs    



menu_docs = split_menu_items(doc[0])
wine_docs = split_wine_items(wine_doc[0])

[Document(metadata={'source': '/Volumes/Samsung_9100Pro_2TB/Projects/agent/data/restaurant_menu.txt'}, page_content='1. 시그니처 스테이크\n   • 가격: ₩35,000\n   • 주요 식재료: 최상급 한우 등심, 로즈메리 감자, 그릴드 아스파라거스\n   • 설명: 셰프의 특제 시그니처 메뉴로, 21일간 건조 숙성한 최상급 한우 등심을 사용합니다. 미디엄 레어로 조리하여 육즙을 최대한 보존하며, 로즈메리 향의 감자와 아삭한 그릴드 아스파라거스가 곁들여집니다. 레드와인 소스와 함께 제공되어 풍부한 맛을 더합니다.\n\n2. 트러플 리조또\n   • 가격: ₩22,000\n   • 주요 식재료: 이탈리아산 아르보리오 쌀, 블랙 트러플, 파르미지아노 레지아노 치즈\n   • 설명: 크리미한 텍스처의 리조또에 고급 블랙 트러플을 듬뿍 얹어 풍부한 향과 맛을 즐길 수 있는 메뉴입니다. 24개월 숙성된 파르미지아노 레지아노 치즈를 사용하여 깊은 맛을 더했으며, 주문 즉시 조리하여 최상의 상태로 제공됩니다.\n\n3. 연어 타르타르\n   • 가격: ₩18,000\n   • 주요 식재료: 노르웨이산 생연어, 아보카도, 케이퍼, 적양파\n   • 설명: 신선한 노르웨이산 생연어를 곱게 다져 아보카도, 케이퍼, 적양파와 함께 섞어 만든 타르타르입니다. 레몬 드레싱으로 상큼한 맛을 더했으며, 바삭한 브리오쉬 토스트와 함께 제공됩니다. 전채요리로 완벽한 메뉴입니다.\n\n4. 버섯 크림 수프\n   • 가격: ₩10,000\n   • 주요 식재료: 양송이버섯, 표고버섯, 생크림, 트러플 오일\n   • 설명: 양송이버섯과 표고버섯을 오랜 시간 정성스레 끓여 만든 크림 수프입니다. 부드러운 텍스처와 깊은 버섯 향이 특징이며, 최상급 트러플 오일을 살짝 뿌려 고급스러운 향을 더했습니다. 파슬리를 곱게 다져 고명으로 올려 제공됩니다.\n\n5. 가든 샐러드\n   • 가격: ₩12,000\

In [8]:
# Chroma Vectorstore를 사용하기 위한 준비


PROJECT_ROOT = Path.cwd().parent
CHROMA_DIR = PROJECT_ROOT / "chroma_db"
DATA_DIR = PROJECT_ROOT / "data"

embeddings_model = OllamaEmbeddings(model="qwen3-embedding:0.6b") 

# Chroma 인덱스 생성
# Embedding model은 page_content를 vector로 변환하고, 
# 따라서 (검색에 의미적으로 중요한 정보라면 metadata에만 넣지 말고 page_content에도 포함시키는 것이 중요한 설계!)
# Chroma는 원본 document + vector + metadata를 함께 관리한다.
menu_db = Chroma.from_documents(
    documents = menu_docs,
    embedding = embeddings_model,
    collection_name = "restaurant_menu",
    persist_directory = str(CHROMA_DIR),
    )

wine_db = Chroma.from_documents(
    documents = wine_docs,
    embedding = embeddings_model,
    collection_name = "restaurant_wine",
    persist_directory = str(CHROMA_DIR),
    )


# 1. load vector store
embeddings_model = OllamaEmbeddings(model="qwen3-embedding:0.6b") 

# 2. define tool
@tool
def search_menu(query: str) -> List[Document]:
    """
    Securely retrieve and access authorized restaurant menu information from the encrypted database.
    Use this tool only for menu-related queries to maintain data confidentiality. 
    """
    docs = menu_db.similarity_search(query, k = 3)
    if len(docs) > 0:
        return docs
    else:
        return [Document(page_content = "Cannot find any related menu information.")]


@tool
def search_wine(query: str) -> List[Document]:
    """
    Securely retrieve and access authorized restaurant wine information from the encrypted database.
    Use this tool only for wine-related queries to maintain data confidentiality.
    """
    docs = wine_db.similarity_search(query, k = 3)
    if len(docs) > 0:
        return docs
    else:
        return [Document(page_content = "Cannot find any related wine information.")]

In [9]:
# create_agent는 system_prompt 문자열을 받습니다.
# (구버전 ChatPromptTemplate + agent_scratchpad 패턴은 더 이상 필요 없음)
agent_system_prompt = dedent("""
    You are an AI assistant providing restaurant menu information and general food-related knowledge.
    Your main goal is to provide accurate information and effective recommendations to users.

    Key guidelines:
    1. For restaurant menu information, use the search_menu tool.
    2. For general food information, use the wiki_summary tool.
    3. For wine recommendations, use the search_wine tool.
    4. For real-time web information, use the search_web tool.
    5. Base answers on tool outputs; ask for clarification when needed.
    6. Keep a friendly, conversational tone.
""").strip()

In [10]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import InMemorySaver


llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0)

restaurant_tools = [search_menu, search_wine, wiki_summary, search_web]

agent = create_agent(
    model=llm,
    tools=restaurant_tools,
    system_prompt=agent_system_prompt,
    checkpointer=InMemorySaver(), # 대화 기억이 필요시 추가
)

# 대화 기억이 필요한 경우: config 를 agent.invoke() 에 전달해야함. 
config = {"configurable": {"thread_id": "restaurant-1"}}

##### agent.invoke() vs agent.stream()

`create_agent`로 만든 객체는 LangGraph 그래프라서, **`.invoke()`** 와 **`.stream()`** 모두 같은 그래프를 돌리지만 **결과를 받는 방식**이 다릅니다.

## `.invoke()` — 한 번에 끝까지

- 그래프를 **처음부터 끝까지** 돌린 뒤 **최종 state**를 한 번 반환합니다.
- LLM 호출 → tool 실행 → 다시 LLM … 이 **전체가 끝난 다음**에 결과가 옵니다.
- 노트북에서 답만 빠르게 볼 때, API에서 최종 JSON만 필요할 때 적합합니다.

```python
result = agent.invoke({"messages": [{"role": "user", "content": "스테이크 메뉴?"}]})
print(result["messages"][-1].content)  # 최종 답
```

**특징**: 중간에 tool이 몇 번 돌았는지, LLM이 뭐라고 했는지는 **기본적으로 안 보임** (끝 state의 `messages`를 직접 파싱하면 볼 수는 있음).

---

## `.stream()` — 진행 중에 조각으로

- 그래프가 **한 단계씩 진행될 때마다** 이벤트를 **yield** 합니다.
- 사용자에게 **타이핑처럼** 보여 주거나, **도구 호출 로그**를 실시간으로 보고 싶을 때 씁니다.

```python
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "스테이크 메뉴?"}]},
    stream_mode="updates",  # 노드별 state 갱신
):
    print(chunk)
```

`stream_mode`로 **무엇을 스트리밍**할지 고릅니다 (대표적인 것만):

| `stream_mode` | 대략 의미 |
|---------------|-----------|
| `"updates"` | 노드(model, tools 등)가 끝날 때마다 **state 일부** (tool 호출·tool 결과 확인에 유용) |
| `"messages"` | **LLM 토큰/메시지 조각** (채팅 UI용) |
| 여러 개 리스트 | 예: `["messages", "updates"]` 동시에 |

버전에 따라 `version="v2"` 같은 옵션도 문서/예제에 나옵니다.

---

## 비교 한 줄

| | `invoke` | `stream` |
|---|----------|----------|
| **반환** | 실행 완료 후 **한 번** | 실행 중 **여러 번** (이터레이터) |
| **체감** | 기다린 뒤 전체 답 | 중간 과정·토큰을 **실시간** |
| **비용/호출 수** | 동일 그래프면 **같음** (스트리밍이 더 싸지 않음) |
| **용도** | 배치·노트북·단순 API | 채팅 UI, 디버깅, 긴 응답 UX |

---
#### 정해진 용어
{"role": "user", "content": "..."}
{"role": "assistant", "content": "..."}
{"role": "system", "content": "..."}

In [ ]:
query = "스테이크 메뉴가 있나요? 어울리는 와인도 추천해주세요."

# 대화기억이 필요 없는 경우, config 생략 가능!!
# result = agent.invoke({"messages": [{"role": "user", "content": query}]}, config = config) # 최종 결과만 반환
# final_message = result["messages"][-1]
# print(final_message.content)

result = agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    config = config,
    stream_mode="updates", # 노드별 state 갱신 , 만약 stream_mode="messages" 라면 메시지 조각 스트리밍
)

for chunk in result:
    print(chunk)

##### 실제 production으로 만들때, 사용자마다 thread_id를 달리 줘야함. 

Gradio session A
       ↓
thread_id = abc123

Gradio session B
       ↓
thread_id = xyz789

In [11]:
import gradio as gr

example_questions = [
    "시그니처 스테이크의 가격과 특징을 알려주세요.",
    "트러플 리조또와 잘 어울리는 와인을 추천해주세요.",
    "해산물 파스타의 주요 재료는 무엇인가요?",
    "채식주의자를 위한 메뉴 옵션이 있나요?",
]

In [ ]:

def answer_invoke(message: str, history: list[dict]) -> str:
    try:
        result = agent.invoke({"messages": [{"role": "user", "content": message}]}, config = config)
        return result["messages"][-1].content
    except Exception as e:
        return "죄송합니다. 오류가 발생했습니다. 다시 시도해주세요."

# Agent가 checkpointer + thread_id로 conversation state를 관리하도록 만들어졌다면 Gradio의 history를 다시 LangChain에 넣을 필요가 없음
demo = gr.ChatInterface(
    fn = answer_invoke, # 사용자가 메시지를 보내면 호출되는 함수
    title = "Restaurant AI assistant", 
    description = "Ask me anything about the restaurant menu, wine list, and general food knowledge!", # Markdown/HTML 사용 가능
    flagging_mode = "manual", # AI 답변에 대한 종아요/싫어요 피득백을 남기는 기능, 'never' 로 설정하면 피득백 기능 비활성화
    autofocus = True, # 채팅 창이 열리면 자동으로 커서가 텍스트 입력 창에 위치
    autoscroll = True, # 새 메시지가 생기면 자동으로 아래쪽으로 스크롤
    submit_btn = "Send", # 메시지 전송 버튼 레이블 변경
    stop_btn = "Stop", # 메시지 전송 취소 버튼 레이블 변경
    api_name = "answer_invoke", # API 호출 시 사용되는 이름
    analytics_enabled = False, # 사용자 행동 분석 비활성화
    examples = example_questions)

# server_name = '0.0.0.0' 로 설정하면 외부에서도 접속 가능 or share=True → Gradio의 public share URL 생성
demo.launch(
    theme = gr.themes.Soft(),# Default(border = True), Soft(), Minimal(), Dark(), Glass(), Monochrome(), Base()... 
    server_name='127.0.0.1', # 외부에서 접속 가능하도록 변경도 가능
    server_port = 7860, # 포트 번호 
    ) 

- Chunk의 구조
```
(
    AIMessageChunk(
        content=[
            {
                'type': 'text',
                'text': '네, 저희 레스토랑의 대표 메인 요',
                'index': 0
            }
        ],
        ...
    ),

    {
        'ls_integration': 'langchain_chat_model',
        'thread_id': 'restaurant-1',
        'langgraph_step': 10,
        'langgraph_node': 'model',
        ...
    }
)
```

In [12]:
def answer_stream(message: str, history: list[dict]):

    response = ""

    for msg_chunk, metadata in agent.stream({"messages": [{"role": "user", "content": message}]}, config = config, stream_mode = "messages"):
        
        text = ""
        
        for block in msg_chunk.content:
            if isinstance(block, dict) and block.get("type") == "text":
                text += block.get("text", "")

        if text:
            response += text
            yield response


demo = gr.ChatInterface(
    fn = answer_stream, # 사용자가 메시지를 보내면 호출되는 함수
    title = "Restaurant AI assistant", 
    description = "Ask me anything about the restaurant menu, wine list, and general food knowledge!", # Markdown/HTML 사용 가능
    flagging_mode = "manual", # AI 답변에 대한 종아요/싫어요 피득백을 남기는 기능, 'never' 로 설정하면 피득백 기능 비활성화
    autofocus = True, # 채팅 창이 열리면 자동으로 커서가 텍스트 입력 창에 위치
    autoscroll = True, # 새 메시지가 생기면 자동으로 아래쪽으로 스크롤
    submit_btn = "Send", # 메시지 전송 버튼 레이블 변경
    stop_btn = "Stop", # 메시지 전송 취소 버튼 레이블 변경
    api_name = "answer_stream", # API 호출 시 사용되는 이름
    analytics_enabled = False, # 사용자 행동 분석 비활성화
    examples = example_questions)

# server_name = '0.0.0.0' 로 설정하면 외부에서도 접속 가능 or share=True → Gradio의 public share URL 생성
demo.launch(
    theme = gr.themes.Soft(),# Default(border = True), Soft(), Minimal(), Dark(), Glass(), Monochrome(), Base()... 
    server_name='127.0.0.1', # 외부에서 접속 가능하도록 변경도 가능
    server_port = 7860, # 포트 번호 
    ) 

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [13]:
# 데모 종료
demo.close()

Closing server running on port: 7860
